## Import Packages

- Use `!pip install` to add any missing packages in Colab.

In [ ]:
!pip install PyWavelets

In [ ]:
import pandas as pd
import numpy as np
import scipy.stats as sp
import pywt

In [ ]:
# Mount your google drive
from google.colab import drive
drive.mount('/content/drive')

.

.

.



## Declare the size of dataset and features

In [ ]:
NoOfData    = 180  # 180 Data for each robotic spot-welding condition (Normal, Abnormal)
NoOfSensor  = 3    # 3 Sensor signals: Acceleration, Voltage, Current
NoOfFeature = 10   # 10 Feature types: Max, Min, Mean, RMS, Variance, Skewness, Kurtosis, Crest factor, Shape factor, Impulse factor

NoOfData, NoOfSensor, NoOfFeature

## Load Raw Dataset (360 files)

In [ ]:
for i in range(NoOfData):

    temp_path1 = f'https://github.com/ljwg3000/UNT_MEEN-AI-Fall2026/blob/main/AI_tutorial/Dataset/Normal_{i+1}?raw=true'   # File path of temporary normal data
    temp_path2 = f'https://github.com/ljwg3000/UNT_MEEN-AI-Fall2026/blob/main/AI_tutorial/Dataset/Abnormal_{i+1}?raw=true' # File path of temporary abnormal data

    exec(f"Normal_{i+1}   = pd.read_csv(temp_path1 , sep=',' , header=None)")
    exec(f"Abnormal_{i+1} = pd.read_csv(temp_path2 , sep=',' , header=None)")

## Time Domain Feature Extraction
- 10 features * 3 sensors = 30 features

In [ ]:
# Define RMS function
def rms(x):
    return np.sqrt(np.mean(x**2))

In [ ]:
# Create empty(0) arrays for normal/abnormal feature dataset (time domain)
TimeFeature_Normal   = np.zeros((NoOfSensor*NoOfFeature , NoOfData))
TimeFeature_Abnormal = np.zeros((NoOfSensor*NoOfFeature , NoOfData))

print(TimeFeature_Normal.shape)
print(TimeFeature_Abnormal.shape)

TimeFeature_Normal

In [ ]:
for i in range(NoOfData):

    exec(f"temp_data1 = Normal_{i+1}")
    exec(f"temp_data2 = Abnormal_{i+1}")

    for j in range(NoOfSensor):

        base_index = NoOfFeature * j

        signal_normal   = temp_data1.iloc[:, j+1]
        signal_abnormal = temp_data2.iloc[:, j+1]

        # Normal features
        TimeFeature_Normal[base_index + 0, i] = np.max(signal_normal)
        TimeFeature_Normal[base_index + 1, i] = np.min(signal_normal)
        TimeFeature_Normal[base_index + 2, i] = np.mean(signal_normal)
        TimeFeature_Normal[base_index + 3, i] = rms(signal_normal)
        TimeFeature_Normal[base_index + 4, i] = np.var(signal_normal)
        TimeFeature_Normal[base_index + 5, i] = sp.skew(signal_normal)
        TimeFeature_Normal[base_index + 6, i] = sp.kurtosis(signal_normal)
        TimeFeature_Normal[base_index + 7, i] = np.max(signal_normal) / rms(signal_normal)
        TimeFeature_Normal[base_index + 8, i] = rms(signal_normal) / np.mean(np.abs(signal_normal))
        TimeFeature_Normal[base_index + 9, i] = np.max(signal_normal) / np.mean(np.abs(signal_normal))

        # Abnormal features
        TimeFeature_Abnormal[base_index + 0, i] = np.max(signal_abnormal)
        TimeFeature_Abnormal[base_index + 1, i] = np.min(signal_abnormal)
        TimeFeature_Abnormal[base_index + 2, i] = np.mean(signal_abnormal)
        TimeFeature_Abnormal[base_index + 3, i] = rms(signal_abnormal)
        TimeFeature_Abnormal[base_index + 4, i] = np.var(signal_abnormal)
        TimeFeature_Abnormal[base_index + 5, i] = sp.skew(signal_abnormal)
        TimeFeature_Abnormal[base_index + 6, i] = sp.kurtosis(signal_abnormal)
        TimeFeature_Abnormal[base_index + 7, i] = np.max(signal_abnormal) / rms(signal_abnormal)
        TimeFeature_Abnormal[base_index + 8, i] = rms(signal_abnormal) / np.mean(np.abs(signal_abnormal))
        TimeFeature_Abnormal[base_index + 9, i] = np.max(signal_abnormal) / np.mean(np.abs(signal_abnormal))

### Combine Normal and Abnormal feature arrays

* axis=0: combine rows
* axis=1: combine columns

In [ ]:
TimeFeature = np.concatenate([TimeFeature_Normal, TimeFeature_Abnormal] , axis=1)
TimeFeature.shape

.

.

.



## Frequency Domain Feature Extraction
- 10 features * 8 wavelet levels * 3 sensors = 240 features

In [ ]:
# Wavelet options
MotherWavelet = pywt.Wavelet('haar')   # Mother wavelet
Level   = 8                            # Wavelet decomposition level

In [ ]:
# Create empty(0) arrays for normal/abnormal feature dataset (frequency Domain)
FreqFeature_Normal   = np.zeros(shape=(NoOfSensor*NoOfFeature*Level , NoOfData))
FreqFeature_Abnormal = np.zeros(shape=(NoOfSensor*NoOfFeature*Level , NoOfData))

print(FreqFeature_Normal.shape)
print(FreqFeature_Abnormal.shape)

FreqFeature_Normal

In [ ]:
for i in range(NoOfData):

    # Declare temporary data (only sensor signals)
    exec(f"temp_data1 = Normal_{i+1}.iloc[:, 1:]")
    exec(f"temp_data2 = Abnormal_{i+1}.iloc[:, 1:]")

    # Wavelet decomposition
    Coef1 = pywt.wavedec(temp_data1, MotherWavelet, level=Level, axis=0)
    Coef2 = pywt.wavedec(temp_data2, MotherWavelet, level=Level, axis=0)

    # Frequency domain feature extraction
    for j in range(NoOfSensor):

        for k in range(Level):

            # Base index for rows
            base_index_sensor = NoOfFeature * Level * j
            base_index_level  = NoOfFeature * k

            base_index        = base_index_sensor + base_index_level

            # Select wavelet coefficients
            coef_normal   = Coef1[Level-k]
            coef_abnormal = Coef2[Level-k]

            signal_normal   = coef_normal[:, j]
            signal_abnormal = coef_abnormal[:, j]

            # Normal features
            FreqFeature_Normal[base_index + 0, i] = np.max(signal_normal)
            FreqFeature_Normal[base_index + 1, i] = np.min(signal_normal)
            FreqFeature_Normal[base_index + 2, i] = np.mean(signal_normal)
            FreqFeature_Normal[base_index + 3, i] = rms(signal_normal)
            FreqFeature_Normal[base_index + 4, i] = np.var(signal_normal)
            FreqFeature_Normal[base_index + 5, i] = sp.skew(signal_normal)
            FreqFeature_Normal[base_index + 6, i] = sp.kurtosis(signal_normal)
            FreqFeature_Normal[base_index + 7, i] = np.max(signal_normal) / rms(signal_normal)
            FreqFeature_Normal[base_index + 8, i] = rms(signal_normal) / np.mean(np.abs(signal_normal))
            FreqFeature_Normal[base_index + 9, i] = np.max(signal_normal) / np.mean(np.abs(signal_normal))

            # Abnormal features
            FreqFeature_Abnormal[base_index + 0, i] = np.max(signal_abnormal)
            FreqFeature_Abnormal[base_index + 1, i] = np.min(signal_abnormal)
            FreqFeature_Abnormal[base_index + 2, i] = np.mean(signal_abnormal)
            FreqFeature_Abnormal[base_index + 3, i] = rms(signal_abnormal)
            FreqFeature_Abnormal[base_index + 4, i] = np.var(signal_abnormal)
            FreqFeature_Abnormal[base_index + 5, i] = sp.skew(signal_abnormal)
            FreqFeature_Abnormal[base_index + 6, i] = sp.kurtosis(signal_abnormal)
            FreqFeature_Abnormal[base_index + 7, i] = np.max(signal_abnormal) / rms(signal_abnormal)
            FreqFeature_Abnormal[base_index + 8, i] = rms(signal_abnormal) / np.mean(np.abs(signal_abnormal))
            FreqFeature_Abnormal[base_index + 9, i] = np.max(signal_abnormal) / np.mean(np.abs(signal_abnormal))


print(FreqFeature_Normal.shape)
print(FreqFeature_Abnormal.shape)

FreqFeature_Normal

### Combine Normal and Abnormal feature arrays

* axis=0: combine rows
* axis=1: combine columns

In [ ]:
FreqFeature = np.concatenate([FreqFeature_Normal, FreqFeature_Abnormal] , axis=1)
FreqFeature.shape

.

.

.



## Final Feature Dataset
- (30 Time domain features + 240 Frequency domain features = 270 features)

In [ ]:
Features = np.concatenate([TimeFeature,FreqFeature] , axis=0)

print(Features.shape)
Features

### Convert Array into Data frame format

* Easy to save as data file (csv)

In [ ]:
Features_df = pd.DataFrame(Features)
Features_df

### Save Final Feature Data in Drive (.csv)

In [ ]:
path = '/content/drive/MyDrive/Colab Notebooks/SavedFiles/FeatureData.csv'
Features_df.to_csv(path, sep=',', header=None , index=None)